# XGBoost: strong baselines for tabular data

A tabular prediction task needs a strong benchmark without assuming that more trees always help. Keep training, validation and test roles distinct; let validation guide stopping; then examine both predictive performance and the limits of feature-importance explanations.

Gradient-boosted trees build an ensemble one tree at a time. Each new tree focuses on errors left by the trees before it. XGBoost makes this approach practical with regularization, efficient training, and a scikit-learn-compatible API.

This notebook uses generated data, so it needs no downloads. It covers classification, validation and early stopping, model evaluation, feature importance, regression, cross-validation, and saving a model.

> **Prerequisite:** Install XGBoost in this notebook's Python environment before running the cells. Follow the current instructions at [xgboost.readthedocs.io/en/stable/install.html](https://xgboost.readthedocs.io/en/stable/install.html). The examples use the modern scikit-learn API available in XGBoost 2.1 and later.

## 1. Import the tools

`XGBClassifier` and `XGBRegressor` follow familiar scikit-learn conventions: create an estimator, call `fit`, then use `predict` or other scoring methods. We will use scikit-learn to make reproducible datasets and calculate evaluation metrics.

In [1]:
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split

print('XGBoost version:', xgb.__version__)

XGBoost version: 2.1.4


## 2. Reserve independent evidence before growing trees

Identify which split learns parameters, which chooses stopping and which estimates final performance. Check class proportions and row counts. Generated data makes the workflow reproducible but does not demonstrate usefulness on a real population.

The generated dataset has 12 numeric features and two target classes. We split it into training, validation, and test sets. The model learns from the training set, uses the validation set to decide when to stop, and is evaluated on the held-out test set. Keeping these roles separate gives a more honest estimate of performance.

In [2]:
features, labels = make_classification(
    n_samples=1600,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    weights=[0.65, 0.35],
    random_state=42,
)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    features, labels, test_size=0.2, stratify=labels, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))
print('Test rows:', len(X_test))
print('Feature count:', X_train.shape[1])

Training rows: 1024
Validation rows: 256
Test rows: 320
Feature count: 12


## 3. Stop adding trees when validation stops improving

Predict how smaller learning rates could affect the number of useful rounds. Inspect the stopping information and relate it to the validation metric. More training rounds are not evidence of better held-out predictions.

`n_estimators` sets the maximum number of boosting rounds. Early stopping watches the validation metric and ends training when it has not improved for the chosen number of rounds. It helps avoid spending time adding trees that no longer improve validation performance.

The `learning_rate` controls how much each new tree contributes. Smaller values often need more trees. `max_depth` limits the complexity of individual trees.

In [3]:
classifier = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective='binary:logistic',
    eval_metric='logloss',
    early_stopping_rounds=25,
    random_state=42,
    n_jobs=-1,
)

classifier.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)

print('Best boosting round:', classifier.best_iteration)

Best boosting round: 269


## 4. Evaluate the kinds of errors the decision would create

Compare accuracy with per-class precision and recall, then inspect ranking performance. A high AUC does not choose a decision threshold or specify the cost of mistakes. Treat the random-forest test comparison as reporting, not permission to tune repeatedly on the test set.

Accuracy is the fraction of correct predictions. ROC AUC summarizes how well the model ranks positive examples above negative ones across classification thresholds. A classification report adds per-class precision, recall, and F1 score. These metrics describe different aspects of performance, so choose them based on the cost of different mistakes.

In [4]:
test_predictions = classifier.predict(X_test)
test_probabilities = classifier.predict_proba(X_test)[:, 1]

print(f'Test accuracy: {accuracy_score(y_test, test_predictions):.3f}')
print(f'Test ROC AUC:  {roc_auc_score(y_test, test_probabilities):.3f}')
print()
print(classification_report(y_test, test_predictions, digits=3))

Test accuracy: 0.900
Test ROC AUC:  0.953

              precision    recall  f1-score   support

           0      0.911     0.937     0.924       207
           1      0.879     0.832     0.855       113

    accuracy                          0.900       320
   macro avg      0.895     0.885     0.889       320
weighted avg      0.899     0.900     0.899       320



As a comparison point, we can fit a random forest on the same training data. This is not a universal contest: model choice should ultimately be based on validation, appropriate metrics, and the constraints of the real problem.

In [10]:
forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
forest.fit(X_train, y_train)
forest_predictions = forest.predict(X_test)

print(f'XGBoost test accuracy:       {accuracy_score(y_test, test_predictions):.3f}')
print(f'Random forest test accuracy: {accuracy_score(y_test, forest_predictions):.3f}')

XGBoost test accuracy:       0.900
Random forest test accuracy: 0.909


## 5. Separate predictive importance from a causal explanation

Inspect which features the fitted model uses most. Ask whether correlated features could share or substitute for that information. Importance is specific to this data and model; it does not establish the effect of intervening on a feature.

The built-in importance scores summarize how the fitted trees used each feature. They are useful for a first inspection, but they do not prove that a feature causes the target or that the model relies on it consistently. For deeper analysis, compare with permutation importance or SHAP values, especially when features are correlated.

In [6]:
importance_order = np.argsort(classifier.feature_importances_)[::-1]

for feature_index in importance_order[:8]:
    importance = classifier.feature_importances_[feature_index]
    print(f'Feature {feature_index:>2}: {importance:.3f}')

Feature  6: 0.130
Feature  7: 0.120
Feature  0: 0.115
Feature  3: 0.110
Feature  1: 0.101
Feature  2: 0.090
Feature 10: 0.086
Feature 11: 0.065


## 6. Use cross-validation for a steadier estimate

A single validation split can be unusually easy or difficult. Cross-validation repeats training and evaluation across several folds. For this quick example we use a smaller estimator without early stopping; each fold is scored on data not used to fit that fold's model.

In [7]:
cross_validation_model = XGBClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.08,
    eval_metric='logloss',
    random_state=42,
    n_jobs=1,
)

fold_scores = cross_val_score(
    cross_validation_model, X_train_full, y_train_full, cv=3, scoring='accuracy', n_jobs=1
)
print('Fold accuracies:', np.round(fold_scores, 3))
print(f'Mean accuracy: {fold_scores.mean():.3f} +/- {fold_scores.std():.3f}')

Fold accuracies: [0.874 0.855 0.897]
Mean accuracy: 0.875 +/- 0.017


## 7. Fit a regression model

XGBoost also handles continuous targets. This synthetic example uses root mean squared error (RMSE), which expresses typical error in the same units as the target. The square root is applied explicitly so the example works across a range of scikit-learn versions.

In [8]:
regression_features, regression_targets = make_regression(
    n_samples=1200, n_features=10, n_informative=6, noise=15.0, random_state=21
)
X_reg_train_full, X_reg_test, y_reg_train_full, y_reg_test = train_test_split(
    regression_features, regression_targets, test_size=0.2, random_state=21
)
X_reg_train, X_reg_valid, y_reg_train, y_reg_valid = train_test_split(
    X_reg_train_full, y_reg_train_full, test_size=0.2, random_state=21
)

regressor = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=25,
    random_state=42,
    n_jobs=-1,
)
regressor.fit(
    X_reg_train, y_reg_train, eval_set=[(X_reg_valid, y_reg_valid)], verbose=False
)

regression_predictions = regressor.predict(X_reg_test)
rmse = mean_squared_error(y_reg_test, regression_predictions) ** 0.5
print('Best boosting round:', regressor.best_iteration)
print(f'Test RMSE: {rmse:.2f}')

Best boosting round: 346
Test RMSE: 27.02


## 8. Save and reload a model

XGBoost's JSON format stores the model structure and learned parameters. It can be loaded later without repeating training, provided the XGBoost package is available. Saving the model is different from saving the entire Python environment or the data preprocessing steps, which should also be tracked in a production workflow.

In [9]:
model_path = 'xgboost_classifier.json'
classifier.save_model(model_path)

restored_classifier = XGBClassifier()
restored_classifier.load_model(model_path)
restored_predictions = restored_classifier.predict(X_test)

print('Predictions match after reload:', np.array_equal(test_predictions, restored_predictions))

Predictions match after reload: True


## Recap

- **Boosting rounds** add trees sequentially to improve the ensemble.
- **Validation sets and early stopping** help choose when to stop training.
- **Held-out test data** provides a final estimate after model choices are made.
- **XGBoost's scikit-learn API** works with familiar metrics and cross-validation tools.
- **Feature importance** is a useful inspection tool, not a causal explanation.
- **JSON model files** preserve a trained model, but not the whole production pipeline.

## Explain what the evidence supports

Report the stopping rule, evaluation split and relevant metric. Explain why one important feature is worth investigating without claiming that it causes the outcome.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: test whether extra tree depth earns its cost

A subscription-scoring export has repeated rows, malformed and missing usage, missing labels, a new category appearing late, and a tempting field populated only after the outcome. Build a time-ordered development/test workflow on 1,000 generated records with a mild late-period shift. This is a synthetic operational scenario, not a real churn dataset; it is designed to expose boundary mistakes while keeping training small.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a feature that predicts too well

`refund_after_outcome` is generated from the label. It would look highly useful but is unavailable when a churn prediction is made. Exclude it by an explicit feature allowlist before preprocessing or tuning. Also exclude row IDs, which index time but are not intended predictors here. Missing labels cannot be imputed as though they were ordinary features.

Fit imputation, scaling and categorical vocabulary only on the training period. An unseen late category should be handled deliberately rather than silently refitting the encoder on the test set.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(63);p2_n=1000
p2_usage=p2_rng.gamma(2,3,p2_n);p2_tenure=p2_rng.integers(1,36,p2_n)
p2_region=p2_rng.choice(['East','West','North'],p2_n).astype(object);p2_region[900:]='NewRegion'
p2_logit=1.8-.28*p2_usage-.035*p2_tenure+.5*(p2_region=='West')+.3*(np.arange(p2_n)>=800)
p2_target=p2_rng.binomial(1,1/(1+np.exp(-p2_logit)))
p2_raw=pd.DataFrame({'row_id':np.arange(p2_n),'usage':p2_usage.astype(object),'tenure':p2_tenure,
 'region':p2_region,'churn':p2_target.astype(float),'refund_after_outcome':p2_target+np.random.default_rng(64).normal(0,.01,p2_n)})
p2_raw.loc[10:19,'usage']='malformed';p2_raw.loc[20:39,'usage']=None;p2_raw.loc[3:7,'churn']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().copy();assert p2_clean.row_id.is_unique
p2_clean['usage']=pd.to_numeric(p2_clean.usage,errors='coerce')
p2_missing_label=int(p2_clean.churn.isna().sum());p2_clean=p2_clean.dropna(subset=['churn'])
p2_clean['churn']=p2_clean.churn.astype(int)
p2_train=p2_clean[p2_clean.row_id<600];p2_valid=p2_clean[(p2_clean.row_id>=600)&(p2_clean.row_id<800)];p2_test=p2_clean[p2_clean.row_id>=800]
p2_features=['usage','tenure','region'];assert 'refund_after_outcome' not in p2_features
assert len(p2_train)==595 and len(p2_valid)==len(p2_test)==200
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss,roc_auc_score,average_precision_score

def p2_preprocess():
    return ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),['usage','tenure']),
      ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['region'])],sparse_threshold=0)
def p2_pipeline(estimator):return Pipeline([('prepare',p2_preprocess()),('model',estimator)])

from xgboost import XGBClassifier
p2_transform=p2_preprocess();p2_xt=p2_transform.fit_transform(p2_train[p2_features]);p2_xv=p2_transform.transform(p2_valid[p2_features]);p2_xe=p2_transform.transform(p2_test[p2_features])
p2_candidates={};p2_rows=[]
for p2_depth in [2,5]:
    p2_model=XGBClassifier(n_estimators=180,max_depth=p2_depth,learning_rate=.05,subsample=.8,colsample_bytree=.9,
      objective='binary:logistic',eval_metric='logloss',early_stopping_rounds=15,random_state=9,n_jobs=1,tree_method='hist')
    p2_t=time.perf_counter();p2_model.fit(p2_xt,p2_train.churn,eval_set=[(p2_xv,p2_valid.churn)],verbose=False)
    p2_rows.append({'depth':p2_depth,'best_iteration':p2_model.best_iteration,'validation_log_loss':log_loss(p2_valid.churn,p2_model.predict_proba(p2_xv)[:,1]),'seconds':time.perf_counter()-p2_t})
    p2_candidates[p2_depth]=p2_model
p2_comparison=pd.DataFrame(p2_rows).sort_values(['validation_log_loss','depth']);display(p2_comparison)
p2_selected=p2_candidates[int(p2_comparison.iloc[0].depth)]

### Work through what early stopping saves

The cap is 180 boosting rounds; validation can stop a candidate earlier. Compare best iteration and elapsed fitting time, not just maximum depth. A deeper tree has a larger capacity budget and may overfit before using the full round allowance.

Evaluate the selected candidate and a fixed logistic baseline on the later test once. Save the booster and check reloaded probabilities on the same transformed matrix. The transformer is separate state: a booster file alone does not explain categorical columns or imputed values.

In [ ]:
import joblib
p2_linear=LogisticRegression(max_iter=1000).fit(p2_xt,p2_train.churn)
p2_report=[]
for p2_name,p2_est in [('validation-selected boosted trees',p2_selected),('fixed linear baseline',p2_linear)]:
    p2_prob=p2_est.predict_proba(p2_xe)[:,1]
    p2_report.append({'model':p2_name,'test_log_loss':log_loss(p2_test.churn,p2_prob),'test_auc':roc_auc_score(p2_test.churn,p2_prob)})
p2_selected.save_model(p2_out/'xgboost_project2.json');joblib.dump(p2_transform,p2_out/'xgboost_project2_preprocessor.joblib')
p2_loaded=XGBClassifier();p2_loaded.load_model(p2_out/'xgboost_project2.json')
assert np.allclose(p2_loaded.predict_proba(p2_xe),p2_selected.predict_proba(p2_xe))
p2_report=pd.DataFrame(p2_report);display(p2_report);p2_report.to_csv(p2_out/'xgboost_temporal_report.csv',index=False)

### Explain an unhelpful increase in complexity

If depth 5 uses more time without a better validation loss, the added capacity has not earned its cost on this development window. If the linear baseline wins on test, report that rather than searching again until the trees win. A single small test does not establish a permanent ranking.

**Try:** turn off subsampling in a new development experiment. **Worked reasoning:** hold the split and all other parameters fixed, then compare validation variation across planned seeds. Feature importance on this one fit does not diagnose leakage; the availability audit must happen before fitting.